In [5]:
import rasterio
import geopandas as gpd
from rasterio.mask import mask
import os

def split_raster_by_grid(original_raster, grid_path, output_dir):
    # 读取矢量网格
    grid = gpd.read_file(grid_path)

    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)

    with rasterio.open(original_raster) as src:
        raster_crs = src.crs
        print("📌 原始栅格坐标系:", raster_crs)

        # 坐标系统一
        if grid.crs != raster_crs:
            print("🔄 正在将矢量网格投影为与栅格一致的坐标系...")
            grid = grid.to_crs(raster_crs)

        for _, row in grid.iterrows():
            tile_id = row["ID"]  # 使用 grid 中的 ID 字段
            geom = [row.geometry]

            try:
                out_image, out_transform = mask(src, geom, crop=True)
            except Exception as e:
                print(f"⚠️ 跳过 ID={tile_id}，裁剪失败: {e}")
                continue

            out_meta = src.meta.copy()
            out_meta.update({
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "crs": raster_crs
            })

            out_path = os.path.join(output_dir, f"building_tile_{tile_id}.tif")
            with rasterio.open(out_path, "w", **out_meta) as dst:
                dst.write(out_image)

            print(f"✅ 已保存切块: {out_path}")

# 示例调用
split_raster_by_grid(
    original_raster=r'D:\shanghai workshop\raster\building.tif',
    grid_path=r"D:\shanghai workshop\vector\shanghai_grid_256m_clipped.shp",
    output_dir=r"D:\shanghai workshop\raster\building_tiles"
)

📌 原始栅格坐标系: PROJCS["UTM_Modified",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",121.4348038],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
🔄 正在将矢量网格投影为与栅格一致的坐标系...
✅ 已保存切块: D:\shanghai workshop\raster\building_tiles\building_tile_194.tif
✅ 已保存切块: D:\shanghai workshop\raster\building_tiles\building_tile_195.tif
✅ 已保存切块: D:\shanghai workshop\raster\building_tiles\building_tile_196.tif
✅ 已保存切块: D:\shanghai workshop\raster\building_tiles\building_tile_197.tif
✅ 已保存切块: D:\shanghai workshop\raster\building_tiles\building_tile_198.tif
✅ 已保存切块: D:\shanghai workshop\raster\building_t